# Comparativa CNN: API **Secuencial** vs **Funcional** (Google Colab)
**Objetivo:** entrenar y comparar dos CNN (misma arquitectura y datos) implementadas con la API *Sequential* y la API *Functional* de Keras/TensorFlow sobre MNIST.
- Entrenamiento: primero **Secuencial** (todas las épocas de un jalón) y luego **Funcional**.
- Comparación: tabla + gráficas (loss / accuracy) y predicciones sobre las mismas imágenes.
- Comentarios: cada bloque y líneas clave incluyen referencias (documentación oficial y tutoriales externos) como comentarios dentro del código.

> Nota: pega este archivo `.ipynb` en Google Colab (Subir -> Archivo) y en *Runtime -> Change runtime type* selecciona **GPU** para aprovechar aceleración.  


In [ ]:
# =======================
# 0) Configuración y parámetros
# =======================
# Importes principales (TensorFlow/Keras, numpy, matplotlib, pandas, time)
# Fuente (docs): Keras/TensorFlow guide. https://www.tensorflow.org/guide/keras  citeturn0search11
import tensorflow as tf                              # API principal de TensorFlow (incluye Keras)
import numpy as np                                   # manejo de arrays numéricos
import matplotlib.pyplot as plt                      # graficado de curvas y ejemplos
import pandas as pd                                  # tablas para comparar resultados
import time                                          # medir tiempos de entrenamiento

# Parámetros configurables
BATCH_SIZE = 64
EPOCHS = 20                 # Cambia este valor a gusto. Recomendación inicial: 20 (buen tradeoff para MNIST)
VALIDATION_SPLIT = 0.10
RANDOM_SEED = 42

# Habilitar modo inline para Jupyter/Colab (muestras de graficas)
%matplotlib inline

# Detectar GPUs (si Colab tiene GPU activada). Recomendación: seleccionar Hardware accelerator -> GPU.
# Fuente (Colab GPU guide): https://colab.research.google.com/notebooks/gpu.ipynb  citeturn0search4
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'GPUs detectadas: {len(gpus)}. Se intentará usar GPU para entrenamiento.')
    try:
        for gpu in gpus:
            # Evita que TF consuma toda la memoria GPU inmediatamente
            tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as e:
        print('No se pudo establecer memory growth en GPU:', e)
else:
    print('No se detectó GPU: el entrenamiento se realizará en CPU.')

In [ ]:
# =======================
# 1) Cargar y preprocesar MNIST
# =======================
# Cargar MNIST (toy dataset): 60k train (28x28), 10k test.
# Fuente: tf.keras.datasets.mnist.load_data() docs. https://www.tensorflow.org/api_docs/python/tf/keras/datasets/mnist/load_data  citeturn0search2
from tensorflow.keras.datasets import mnist
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Revisar shapes (opcional), aquí comentado para no spamear en ejecución
# print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)  # (60000,28,28) (60000,) (10000,28,28) (10000,)

# Normalizar (0-1) y agregar canal final para Conv2D (shape -> (N,28,28,1))
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32') / 255.0
X_train = X_train.reshape((-1, 28, 28, 1))
X_test  = X_test.reshape((-1, 28, 28, 1))

# One-hot encoding para la salida (10 clases)
from tensorflow.keras.utils import to_categorical
y_train_cat = to_categorical(y_train, num_classes=10)
y_test_cat  = to_categorical(y_test,  num_classes=10)

# Fijar semilla para reproducibilidad (en la medida de lo posible)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

In [ ]:
# =======================
# 2) Construir y entrenar modelo SECUENCIAL (todo en un jalón)
# =======================
# Usamos tf.keras.Sequential: apropiado para stacks lineales de capas.
# Documentacion Sequential: https://www.tensorflow.org/api_docs/python/tf/keras/Sequential  citeturn0search0
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense

def build_sequential_model():
    # Arquitectura: (misma que en el archivo original que subiste)
    # Input -> Conv2D(64,3x3) -> MaxPool -> Conv2D(128,3x3) -> MaxPool -> Flatten -> Dense(128) -> Dense(10 softmax)
    model = Sequential([
        Input(shape=(28,28,1), name='seq_input'),                     # Capa de entrada explícita
        Conv2D(64, kernel_size=(3,3), activation='relu', name='seq_conv2d_1'),  # Conv1 (64 filtros)
        MaxPooling2D(pool_size=(2,2), name='seq_pool1'),              # Reducción espacial
        Conv2D(128, kernel_size=(3,3), activation='relu', name='seq_conv2d_2'), # Conv2 (128 filtros)
        MaxPooling2D(pool_size=(2,2), name='seq_pool2'),
        Flatten(name='seq_flatten'),
        Dense(128, activation='relu', name='seq_dense128'),
        Dense(10, activation='softmax', name='seq_output')
    ])
    return model

# Construcción y compilación
seq_model = build_sequential_model()
# Compilación: Adam optimizer, categorical_crossentropy loss y metric accuracy
# Fuente model.compile: https://www.tensorflow.org/api_docs/python/tf/keras/Model#compile  citeturn0search16
seq_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Resumen (imprime la arquitectura)
print('--- Resumen Modelo SECUENCIAL ---')
seq_model.summary()

# Entrenamiento: un solo llamado a fit() con EPOCHS definidas arriba.
# Devuelve History con métricas por época.
start_time = time.time()
history_seq = seq_model.fit(
    X_train, y_train_cat,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    verbose=2
)
time_seq = time.time() - start_time

# Evaluación final sobre conjunto de test
loss_seq, acc_seq = seq_model.evaluate(X_test, y_test_cat, verbose=0)
print(f'SECUENCIAL -> loss: {loss_seq:.4f}, accuracy: {acc_seq:.4f}, tiempo(s): {time_seq:.1f}')

# Guardar modelo entrenado (archivo .keras)
seq_model.save('cnn_secuencial.keras')  # se guarda en el workspace de Colab


In [ ]:
# =======================
# 3) Construir y entrenar modelo FUNCIONAL (todo en un jalón)
# =======================
# Functional API: permite topologías más complejas (DAG), múltiples entradas/salidas, etc.
# Fuente: Functional API guide. https://www.tensorflow.org/guide/keras/functional_api  citeturn0search1
from tensorflow.keras import Input as KInput
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D as KConv2D, MaxPooling2D as KMaxPool, Flatten as KFlatten, Dense as KDense

def build_functional_model():
    inputs = KInput(shape=(28,28,1), name='func_input')
    x = KConv2D(64, (3,3), activation='relu', name='func_conv2d_1')(inputs)
    x = KMaxPool((2,2), name='func_pool1')(x)
    x = KConv2D(128, (3,3), activation='relu', name='func_conv2d_2')(x)
    x = KMaxPool((2,2), name='func_pool2')(x)
    x = KFlatten(name='func_flatten')(x)
    x = KDense(128, activation='relu', name='func_dense128')(x)
    outputs = KDense(10, activation='softmax', name='func_output')(x)
    model = Model(inputs=inputs, outputs=outputs, name='cnn_funcional')
    return model

# Construcción y compilación
func_model = build_functional_model()
func_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print('--- Resumen Modelo FUNCIONAL ---')
func_model.summary()

# Entrenamiento
start_time = time.time()
history_func = func_model.fit(
    X_train, y_train_cat,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    verbose=2
)
time_func = time.time() - start_time

loss_func, acc_func = func_model.evaluate(X_test, y_test_cat, verbose=0)
print(f'FUNCIONAL -> loss: {loss_func:.4f}, accuracy: {acc_func:.4f}, tiempo(s): {time_func:.1f}')

# Guardar modelo
func_model.save('cnn_funcional.keras')

In [ ]:
# =======================
# 4) Comparación y visualización de resultados
# =======================
# Tabla resumen con métricas finales y tiempos de entrenamiento
results = pd.DataFrame({
    'Modelo': ['Secuencial', 'Funcional'],
    'Loss_test': [loss_seq, loss_func],
    'Accuracy_test': [acc_seq, acc_func],
    'Tiempo_s': [round(time_seq,1), round(time_func,1)]
})
print('--- Resumen final ---')
display(results)

# Graficar curvas de loss y accuracy para ambos modelos (train/val)
epochs_range = range(1, EPOCHS+1)

plt.figure(figsize=(14,5))

plt.subplot(1,2,1)
plt.plot(epochs_range, history_seq.history['loss'], marker='o', label='Seq train loss')
plt.plot(epochs_range, history_seq.history['val_loss'], marker='o', label='Seq val loss')
plt.plot(epochs_range, history_func.history['loss'], marker='x', label='Func train loss')
plt.plot(epochs_range, history_func.history['val_loss'], marker='x', label='Func val loss')
plt.title('Loss: Secuencial vs Funcional')
plt.xlabel('Época'); plt.ylabel('Loss'); plt.legend(); plt.grid(True)

plt.subplot(1,2,2)
plt.plot(epochs_range, history_seq.history['accuracy'], marker='o', label='Seq train acc')
plt.plot(epochs_range, history_seq.history['val_accuracy'], marker='o', label='Seq val acc')
plt.plot(epochs_range, history_func.history['accuracy'], marker='x', label='Func train acc')
plt.plot(epochs_range, history_func.history['val_accuracy'], marker='x', label='Func val acc')
plt.title('Accuracy: Secuencial vs Funcional')
plt.xlabel('Época'); plt.ylabel('Accuracy'); plt.legend(); plt.grid(True)

plt.tight_layout()
plt.show()

# Mostrar predicciones sobre el mismo conjunto de imágenes para comparar
n_examples = 6
indices = np.random.choice(len(X_test), n_examples, replace=False)
images = X_test[indices]

preds_seq = seq_model.predict(images)
preds_func = func_model.predict(images)

plt.figure(figsize=(12,4))
for i, idx in enumerate(indices):
    ax = plt.subplot(2, n_examples, i+1)
    plt.imshow(images[i].reshape(28,28), cmap='gray')
    plt.title(f'True: {np.argmax(y_test_cat[idx])}')
    plt.axis('off')
    ax2 = plt.subplot(2, n_examples, i+1+n_examples)
    plt.axis('off')
    txt = f'Seq: {np.argmax(preds_seq[i])} ({preds_seq[i].max():.2%})\\nFunc: {np.argmax(preds_func[i])} ({preds_func[i].max():.2%})'
    ax2.text(0.1, 0.5, txt, fontsize=12)
plt.suptitle('Comparación de predicciones - Secuencial vs Funcional', fontsize=14)
plt.show()

# Guardar historiales localmente (json) para análisis posterior
import json
with open('history_seq.json','w') as f:
    json.dump(history_seq.history, f)
with open('history_func.json','w') as f:
    json.dump(history_func.history, f)

print('Archivos guardados: cnn_secuencial.keras, cnn_funcional.keras, history_seq.json, history_func.json')

# Referencias utilizadas (documentación oficial y tutoriales)
Se listan las fuentes principales que incluí como comentarios en el código (docs oficiales y tutoriales externos):

1. TensorFlow / Keras — Sequential API. https://www.tensorflow.org/api_docs/python/tf/keras/Sequential  citeturn0search0
2. TensorFlow — Functional API guide. https://www.tensorflow.org/guide/keras/functional_api  citeturn0search1
3. tf.keras.datasets.mnist.load_data documentation. https://www.tensorflow.org/api_docs/python/tf/keras/datasets/mnist/load_data  citeturn0search2
4. Conv2D layer docs. https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D  citeturn0search3
5. Using GPU in Colab (Colab GPU notebook). https://colab.research.google.com/notebooks/gpu.ipynb  citeturn0search4
6. Comparison article: Sequential vs Functional (Analytics Vidhya / Medium). https://medium.com/analytics-vidhya/keras-model-sequential-api-vs-functional-api-fc1439a6fb10  citeturn0search5
7. Keras datasets overview. https://keras.io/api/datasets/  citeturn0search8
8. Use a GPU (TensorFlow guide). https://www.tensorflow.org/guide/gpu  citeturn0search10
9. Conv2D detailed (Keras): https://keras.io/api/layers/convolution_layers/convolution2d/  citeturn0search9
10. Keras guide overview. https://www.tensorflow.org/guide/keras  citeturn0search11

> Si quieres que añada más artículos (papers o tutoriales específicos), dime cuáles y los incluyo en el notebook como referencias adicionales.
